[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/96_pca_anomaly_oppo260808_solution.ipynb)

# 参考解法：PCA 重构误差异常检测

Reference solution.

## 解析

**结论：PCA 学到正常样本的低维主子空间，正常点能被少数主成分很好地重构（误差小），异常点偏离主子空间导致重构误差大；以训练误差的 95 分位数为阈值判定。**

### 缺失值填补
`np.array(..., dtype=float)` 把 `None` 转成 `nan`。只用**训练集**每列均值 `np.nanmean(train, axis=0)` 填补，`train` 与 `test` 都用同一组均值，避免测试信息泄漏。

### 标准化 + PCA
`StandardScaler` 只在 `train` 上 `fit`，消除量纲差异；`PCA(n_components=0.95)` 保留 95% 方差的主成分。重构 `X_rec = pca.inverse_transform(pca.transform(X))` 是把样本投影到主子空间再还原。

### 误差与阈值
重构误差取每特征平方差之和 `np.sum((X - X_rec)**2, axis=1)`。阈值为训练误差的 95 分位数，`test` 误差**严格大于**阈值判为异常（`1`）。

### 验证
已用独立参考实现在数十组随机 `(train, test)`（含缺失值、不同维度）上逐样本对拍一致，并复现官方样例 `[0, 1, 0]`。

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass

In [ ]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

In [ ]:
# ✅ SOLUTION

import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

def detect_anomaly(train, test):
    X = np.array(train, dtype=float)          # None -> nan
    T = np.array(test, dtype=float)
    means = np.nanmean(X, axis=0)             # train-only column means
    r, c = np.where(np.isnan(X)); X[r, c] = means[c]
    r, c = np.where(np.isnan(T)); T[r, c] = means[c]
    sc = StandardScaler(); Xs = sc.fit_transform(X); Ts = sc.transform(T)
    pca = PCA(n_components=0.95, svd_solver='full'); pca.fit(Xs)
    Xr = pca.inverse_transform(pca.transform(Xs))
    Tr = pca.inverse_transform(pca.transform(Ts))
    etr = np.sum((Xs - Xr) ** 2, axis=1)
    ete = np.sum((Ts - Tr) ** 2, axis=1)
    thr = np.percentile(etr, 95)
    return (ete > thr).astype(int).tolist()

In [ ]:
train = [[1, 2], [2, 4.1], [3, 5.9], [4, 8.1], [5, 10]]
test = [[2.5, 5.0], [2.5, 10.0], [3.0, None]]
print(detect_anomaly(train, test))   # [0, 1, 0]

In [ ]:
from torch_judge import check
check('pca_anomaly_oppo260808')